In [ ]:
# One-cell Phase10/11 original 646-assay QTL scalar N sensitivity (min/median/first).
# Input: original archived BBJ x GTEx V8 four-locus screen; NOT 80 new windows.
from pathlib import Path
import csv
import hashlib
import json
import shutil
import subprocess
import time
from google.colab import drive
drive.mount("/content/drive")
WORK = Path("/content/is_phase10_11_646_qtl_n_stress")
WORK.mkdir(parents=True,exist_ok=True)
GDRIVE = Path("/content/drive/MyDrive/MASTER_DEGREE/IS_COLAB")
INDEX = GDRIVE/"coloc_ready_v2/COLOC_ABF_INPUT_INDEX_V2.tsv"
MASTER = GDRIVE/"results_v2/COLOC_ABF_MASTER_V2.tsv"
INPUT = GDRIVE/"coloc_ready_v2/abf"
OUTPUT = GDRIVE/"results/IS_PHASE10_11_646_QTL_N_SCALAR_SENSITIVITY_V1"
RUNLOG = WORK/"IS_PHASE10_11_646_QTL_N_R_LOG.txt"
for p in [INDEX, MASTER, INPUT]:
    if not p.exists():
        raise FileNotFoundError(f"Missing original Drive source: {p}")
if OUTPUT.exists():
    raise FileExistsError(f"Refusing existing result folder: {OUTPUT}. Use new version suffix after reviewing past results.")
def sha(path):
    h=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(8*1024*1024),b""):
            h.update(block)
    return h.hexdigest()
with INDEX.open(encoding="utf8",newline="") as handle:
    source_rows=list(csv.DictReader(handle,delimiter="\t"))
if len(source_rows)!=646:
    raise ValueError(f"Expected original 646 index rows, got {len(source_rows)}")
input_files=[INPUT/Path(x["file"]).name for x in source_rows]
n_available=sum(p.exists() and p.stat().st_size>0 for p in input_files)
print("INPUTS_INDEXED=",len(input_files))
print("INPUTS_AVAILABLE_ON_DRIVE=",n_available)
print("INPUTS_MISSING_ON_DRIVE=",646-n_available)
print("INDEX_SHA256=",sha(INDEX))
print("MASTER_SHA256=",sha(MASTER))
assert sha(INDEX)=="1fe044f1ffd8e9aa6b678718b6772063bd11edd462d7bede4104843cdb0c7414", "INDEX_SOURCE_SHA_CHANGED"
assert sha(MASTER)=="0160a01fc6433f9bec839113ac3a2390927fc07db9252d014b0cc4b6b11716ea", "MASTER_SOURCE_SHA_CHANGED"
print("MEMORY=",Path("/proc/meminfo").read_text().splitlines()[0])
if shutil.which("Rscript") is None:
    subprocess.run(["apt-get","update","-qq"],check=True)
    subprocess.run(["apt-get","install","-y","-qq",
                    "r-base","r-base-dev","build-essential",
                    "libcurl4-openssl-dev","libssl-dev","libxml2-dev",
                    "r-cran-data.table"],check=True)
R_INSTALL = r"""
repos <- "https://cloud.r-project.org"
if (!requireNamespace("data.table",quietly=TRUE)) {
  install.packages("data.table",repos=repos,Ncpus=2)
}
ok <- requireNamespace("coloc",quietly=TRUE) &&
      as.character(packageVersion("coloc"))=="5.2.3"
if (!ok) {
  deps <- c("ggplot2","susieR","mixsqp","Rcpp","viridis")
  need <- deps[!vapply(deps,requireNamespace,logical(1),quietly=TRUE)]
  if (length(need)>0) install.packages(need,repos=repos,Ncpus=2)
  # Prefer CRAN current exact source; fall back to its archived release.
  candidates <- c(
    "https://cran.r-project.org/src/contrib/coloc_5.2.3.tar.gz",
    "https://cran.r-project.org/src/contrib/Archive/coloc/coloc_5.2.3.tar.gz")
  installed <- FALSE
  for (url in candidates) {
    tryCatch({
      suppressWarnings(install.packages(url,repos=NULL,type="source"))
      installed <- requireNamespace("coloc",quietly=TRUE) &&
        as.character(packageVersion("coloc"))=="5.2.3"
    },error=function(e) cat("PACKAGE_SOURCE_FAILED:",conditionMessage(e),"\n"))
    if (installed) break
  }
  if (!installed) stop("Unable to pin coloc 5.2.3 from CRAN current or archive")
}
stopifnot(requireNamespace("coloc",quietly=TRUE))
stopifnot(requireNamespace("data.table",quietly=TRUE))
stopifnot(as.character(packageVersion("coloc"))=="5.2.3")
cat("COL0C_PINNED_5.2.3_READY\n")
"""
install=WORK/"install_is_646_deps.R"
install.write_text(R_INSTALL,encoding="utf8")
subprocess.run(["Rscript","--vanilla",str(install)],check=True)
R_CODE = '#!/usr/bin/env Rscript\n# Sensitivity of the archived original coloc v5.2.3 first-SNP QTL N assumption.\n# Scalar-N stress test ONLY, not correct per-variant sample size model.\nsuppressPackageStartupMessages({library(data.table);library(coloc)})\nstopifnot(as.character(packageVersion("coloc"))=="5.2.3")\nargv<-commandArgs(trailingOnly=TRUE)\nif(length(argv)!=4L)stop("usage: SCRIPT original_index original_master existing_snp_dir new_output_dir")\nip<-normalizePath(argv[1],mustWork=TRUE)\nmp<-normalizePath(argv[2],mustWork=TRUE)\ndir_in<-normalizePath(argv[3],mustWork=TRUE)\nout<-argv[4]\nif(dir.exists(out))stop("FAIL_CLOSED_EXISTING_OUTPUT")\nidx<-fread(ip);orig<-fread(mp)\nkey<-c("locus","dataset_key","gene_base")\nif(nrow(idx)!=646L||nrow(orig)!=646L||\n   anyDuplicated(idx,by=key)||anyDuplicated(orig,by=key))stop("ORIGINAL_INDEX_SCHEMA_OR_KEYS_INVALID")\njoined<-merge(idx,orig,by=key,sort=FALSE,suffixes=c("_idx","_orig"))\nif(nrow(joined)!=646L||any(joined$status_idx!="READY")||\n   any(joined$status_orig!="PASS")||\n   any(joined$nsnps_idx!=joined$nsnps_orig))stop("INDEX_MASTER_ALIGNMENT_INVALID")\nsetorder(joined,locus,dataset_key,gene_base)\nn_gwas<-174686;s_gwas<-22664/n_gwas\npriors<-list(p1=1e-4,p2=1e-4,p12=1e-5)\nrows<-vector("list",646)\nfor(i in seq_len(nrow(joined))){\n  s<-joined[i]\n  name<-paste0(s$locus,"__",s$dataset_key,"__",s$gene_base,".tsv")\n  if(!identical(basename(s$file),name))stop("INDEX_INPUT_FILE_NAMING_INVALID")\n  file<-file.path(dir_in,name)\n  row<-data.table(locus=s$locus,dataset_key=s$dataset_key,gene_base=s$gene_base,\n                  filename=name,status="MISSING_INPUT",failure_reason="SOURCE_TSV_NOT_LOCALLY_CACHED",\n                  n_snps=NA_integer_,n_qtl_min=NA_real_,n_qtl_median=NA_real_,\n                  n_qtl_first=NA_real_,n_qtl_max=NA_real_,n_qtl_iqr=NA_real_,\n                  fraction_snp_N_below_first=NA_real_,\n                  H4_FIRST=NA_real_,H4_MEDIAN=NA_real_,H4_MIN=NA_real_,\n                  H3_FIRST=NA_real_,H3_MEDIAN=NA_real_,H3_MIN=NA_real_,\n                  H4_cond_FIRST=NA_real_,H4_cond_MEDIAN=NA_real_,H4_cond_MIN=NA_real_,\n                  abs_delta_H4_min=NA_real_,abs_delta_H4_median=NA_real_,\n                  robust_H4_ge_0_5=NA,robust_H4_ge_0_75=NA)\n  if(file.exists(file)){\n    tryCatch({\n      x<-fread(file)\n      req<-c(key,"match_key","eqtl_n","qtl_n_scalar","eqtl_beta","eqtl_se","eqtl_maf","gwas_beta","gwas_se","gwas_maf")\n      if(!all(req%in%names(x))||nrow(x)!=s$nsnps_idx||\n         uniqueN(x$match_key)!=nrow(x))stop("INVALID_SNP_INPUT")\n      if(any(x$locus!=s$locus)||any(x$dataset_key!=s$dataset_key)||\n         any(x$gene_base!=s$gene_base))stop("INPUT_COMPOSITE_KEY_MISMATCH")\n      n<-as.numeric(x$eqtl_n)\n      if(any(!is.finite(n))||any(n<=0))stop("QTL_N_INVALID")\n      if(abs(n[1]-s$qtl_n_first)>1e-8||\n         abs(max(n)-s$qtl_n_max)>1e-8||\n         abs(min(n)-s$qtl_n_min)>1e-8||\n         abs(s$qtl_n_first-s$qtl_n_max)>1e-8||\n         uniqueN(x$qtl_n_scalar)!=1L||\n         abs(x$qtl_n_scalar[1]-n[1])>1e-8)\n         stop("ARCHIVED_QTL_N_RANGE_OR_FIRST_MAX_NOT_REPRODUCED")\n      a<-c(First=n[1],Median=median(n),Min=min(n))\n      d1<-list(beta=as.numeric(x$gwas_beta),varbeta=as.numeric(x$gwas_se)^2,\n               snp=as.character(x$match_key),MAF=as.numeric(x$gwas_maf),\n               N=n_gwas,s=s_gwas,type="cc")\n      h4<-setNames(numeric(3),names(a))\n      h3<-h4\n      for(j in seq_along(a)){\n        d2<-list(beta=as.numeric(x$eqtl_beta),varbeta=as.numeric(x$eqtl_se)^2,\n                 snp=as.character(x$match_key),MAF=as.numeric(x$eqtl_maf),\n                 N=as.numeric(a[j]),type="quant")\n        fit<-suppressMessages(suppressWarnings(coloc.abf(d1,d2,p1=priors$p1,p2=priors$p2,p12=priors$p12)))\n        h4[j]<-as.numeric(fit$summary["PP.H4.abf"])\n        h3[j]<-as.numeric(fit$summary["PP.H3.abf"])\n      }\n      if(any(!is.finite(h4))||any(!is.finite(h3)))stop("NONFINITE_RESULT")\n      if(abs(h4["First"]-s[["PP.H4"]])>1e-8||\n         abs(h3["First"]-s[["PP.H3"]])>1e-8)stop("BASELINE_NOT_REPRODUCED")\n      conditional<-h4/(h4+h3)\n      if(any(!is.finite(conditional)))stop("H4_CONDITIONAL_NOT_DEFINED")\n      row[,\':=\'(n_snps=nrow(x),n_qtl_min=min(n),n_qtl_median=median(n),\n                n_qtl_first=n[1],n_qtl_max=max(n),n_qtl_iqr=IQR(n),\n                fraction_snp_N_below_first=mean(n<n[1]),\n                H4_FIRST=h4["First"],H4_MEDIAN=h4["Median"],H4_MIN=h4["Min"],\n                H3_FIRST=h3["First"],H3_MEDIAN=h3["Median"],H3_MIN=h3["Min"],\n                H4_cond_FIRST=conditional["First"],H4_cond_MEDIAN=conditional["Median"],H4_cond_MIN=conditional["Min"],\n                abs_delta_H4_min=abs(h4["Min"]-h4["First"]),\n                abs_delta_H4_median=abs(h4["Median"]-h4["First"]),\n                robust_H4_ge_0_5=all(h4>=0.5),robust_H4_ge_0_75=all(h4>=0.75),\n                status="PASS",failure_reason="")]\n    },error=function(e){\n      row[,\':=\'(status="FAIL",failure_reason=gsub("[\\\\r\\\\n\\\\t]"," ",conditionMessage(e)))]\n    })\n  }\n  rows[[i]]<-row\n  if(i%%100==0||i==646){\n    cat("N_SCALAR_PROGRESS",i,"/646 PASS",sum(vapply(rows[seq_len(i)],function(z)z$status=="PASS",logical(1))),"\\n")\n  }\n}\nres<-rbindlist(rows,fill=TRUE)\nif(nrow(res)!=646||anyDuplicated(res,by=key)||\n   sum(res$status%in%c("PASS","FAIL","MISSING_INPUT"))!=646)stop("STATUS_PARTITION_FAILED")\npass<-res[status=="PASS"]\nsummary<-data.table(\n   metric=c("total_original_test_count","numeric_sensitivity_PASS","numeric_sensitivity_FAIL",\n            "inputs_not_cached","n_first_equals_max","n_median_equals_first",\n            "n_baseline_H4_ge_0_5","n_min_H4_ge_0_5",\n            "n_baseline_H4_ge_0_75","n_min_H4_ge_0_75",\n            "n_threshold_0_5_changed","n_threshold_0_75_changed",\n            "max_abs_delta_H4_min","median_abs_delta_H4_min"),\n   value=as.character(c(nrow(res),nrow(pass),sum(res$status=="FAIL"),\n            sum(res$status=="MISSING_INPUT"),\n            sum(pass$n_qtl_first==pass$n_qtl_max),\n            sum(pass$n_qtl_median==pass$n_qtl_first),\n            sum(pass$H4_FIRST>=.5),sum(pass$H4_MIN>=.5),\n            sum(pass$H4_FIRST>=.75),sum(pass$H4_MIN>=.75),\n            sum((pass$H4_FIRST>=.5)!=(pass$H4_MIN>=.5)),\n            sum((pass$H4_FIRST>=.75)!=(pass$H4_MIN>=.75)),\n            if(nrow(pass))max(pass$abs_delta_H4_min)else NA_real_,\n            if(nrow(pass))median(pass$abs_delta_H4_min)else NA_real_))\n)\ndir.create(out,recursive=TRUE,showWarnings=FALSE)\nfwrite(res,file.path(out,"IS_646_QTL_N_SCALAR_SENSITIVITY.tsv"),sep="\\t",na="NA")\nfwrite(summary,file.path(out,"IS_646_QTL_N_SCALAR_SUMMARY.tsv"),sep="\\t")\nwriteLines(c(\n "# IS original-646 QTL scalar-N stress test",\n "",\n "- Uses unmodified archived SNP beta/SE/MAF and coloc v5.2.3 priors, and compares first/max N versus median N versus minimum SNP N.",\n "- These are **three scalar N approximations**. They do not implement per-SNP effective N correction, estimate phenotype variance from donor genotypes, or provide biological replication.",\n "- All original 646 index rows are reported with PASS/FAIL/MISSING_INPUT and source file key; only actual cached input TSVs can be PASS.",\n "- Scalar N sensitivity can be weak because coloc.abf quantitative-trait sdY approximation uses both per-SNP SE and MAF; no change does not establish correct n_eff.",\n "- Variant effect-allele alignment and GTEx cohort-matched LD remain separate unresolved issues.",\n paste0("- PASS=",nrow(pass),", FAIL=",sum(res$status=="FAIL"),\n        ", MISSING_INPUT=",sum(res$status=="MISSING_INPUT")," (local snapshot only).")\n),file.path(out,"IS_646_QTL_N_SCALAR_INTERPRETATION.md"))\ncat("IS_QTL_N_SCALAR_SENSITIVITY",if(sum(res$status=="FAIL")==0)"NO_COMPUTATION_FAILURE"else"FAILURE_PRESENT",\n "PASS",nrow(pass),"FAIL",sum(res$status=="FAIL"),"MISSING",sum(res$status=="MISSING_INPUT"),"\\n")\n'
script=WORK/"IS_PHASE10_11_646_qtl_n.R"
script.write_text(R_CODE,encoding="utf8")
with script.open(encoding="utf8") as f:
    print("R_SCRIPT_SHA256=",sha(script))
subprocess.run(["Rscript","--vanilla","-e",
  "invisible(parse(file=commandArgs(trailingOnly=TRUE)[1]));cat('QTL_N_SENSITIVITY_R_PARSE_OK\\n')",
  str(script)],check=True)
start=time.time()
with RUNLOG.open("w",encoding="utf8") as log:
    proc=subprocess.run(["Rscript","--vanilla",str(script),
          str(INDEX),str(MASTER),str(INPUT),str(OUTPUT)],
          stdout=log,stderr=subprocess.STDOUT,check=False)
print("COLOC_PROCESS_EXIT=",proc.returncode)
print("RUNTIME_SECONDS=",round(time.time()-start,2))
lines=RUNLOG.read_text(encoding="utf8",errors="replace").splitlines()
print("\n".join(lines[-35:]))
if proc.returncode!=0:
    raise RuntimeError(f"R replay failed (exit={proc.returncode}); inspect {RUNLOG}. Original files unchanged.")
status=OUTPUT/"IS_646_QTL_N_SCALAR_SENSITIVITY.tsv"
if not status.exists():
    raise FileNotFoundError("Runner emitted no audit status")
with status.open(encoding="utf8",newline="") as f:
    results=list(csv.DictReader(f,delimiter="\t"))
if len(results)!=646:
    raise ValueError("Final status must include exactly 646 original rows")
counts={k:sum(row["status"]==k for row in results) for k in ("PASS","FAIL","MISSING_INPUT")}
print("N_SCALAR_646_SUMMARY_FILE=",OUTPUT/"IS_646_QTL_N_SCALAR_SUMMARY.tsv")
if sum(counts.values())!=646:
    raise ValueError("Status counts do not sum to 646")
# Hash each accessible SNP input after the audit. The output remains a fresh folder.
with (OUTPUT/"IS_646_QTL_N_SOURCE_SHA256.tsv").open("w",encoding="utf8",newline="") as audit_file:
    writer=csv.writer(audit_file,delimiter="\t")
    writer.writerow(["locus","dataset_key","gene_base","filename","status","source_sha256"])
    for entry in results:
        source_input=INPUT/entry["filename"]
        file_hash=sha(source_input) if source_input.is_file() else "MISSING"
        writer.writerow([entry["locus"],entry["dataset_key"],entry["gene_base"],
                        entry["filename"],entry["status"],file_hash])
G1_CODE = '#!/usr/bin/env python3\n"""IS original 646 GTEx/GWAS variant source G1 structural harmonization audit.\n\nExact GRCh38 variant identity/GTEx fields and GWAS-MAF internal arithmetic.\nDoes not prove GWAS effect allele orientation or GRCh37->38 liftover.\nNo changes to input data. Reports missing local source, never PASS.\n"""\nimport argparse\nimport collections\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\ndef data(path):\n    with path.open(encoding="utf8",newline="") as f:\n        return list(csv.DictReader(f,delimiter="\\t"))\n\ndef filehash(p):\n    sha=hashlib.sha256()\n    with p.open("rb") as f:\n        for b in iter(lambda:f.read(1024*1024),b""):\n            sha.update(b)\n    return sha.hexdigest()\n\ndef parse_record(z):\n    fields=("variant","chromosome","position","ref","alt","match_key",\n            "variant_id_hg38","gwas_eaf","gwas_maf","eqtl_maf","maf",\n            "eqtl_beta","beta","eqtl_se","se","harmonization","variant_id")\n    if any(k not in z or z[k]=="" for k in fields):\n        raise ValueError("SOURCE_ALLELE_REQUIRED_FIELD_MISSING")\n    ch=z["chromosome"].removeprefix("chr")\n    pos=int(z["position"])\n    ref=z["ref"].upper()\n    alt=z["alt"].upper()\n    if ref==alt or any(x not in "ACGT" for x in ref+alt):\n        raise ValueError("QTL_ALLELE_NONSTANDARD_OR_IDENTICAL")\n    target=f"{ch}:{pos}:{ref}:{alt}"\n    v=f"chr{ch}_{pos}_{ref}_{alt}"\n    if z["variant"]!=v or z["match_key"]!=target or z["variant_id_hg38"]!=target:\n        raise ValueError("GTEX_GRCH38_ALLELE_IDENTITY_MISMATCH")\n    if z["harmonization"]!="EXACT_REF_ALT_GRCH38":\n        raise ValueError("SOURCE_HARMONIZATION_TAG_UNEXPECTED")\n    if abs(float(z["eqtl_beta"])-float(z["beta"]))>1e-9 or abs(float(z["eqtl_se"])-float(z["se"]))>1e-9:\n        raise ValueError("EQTL_BETA_SE_NOT_SOURCE")\n    if abs(float(z["eqtl_maf"])-float(z["maf"]))>1e-9:\n        raise ValueError("EQTL_MAF_NOT_SOURCE")\n    eaf=float(z["gwas_eaf"])\n    gmaf=float(z["gwas_maf"])\n    qmaf=float(z["eqtl_maf"])\n    if any(not math.isfinite(x) for x in (eaf,gmaf,qmaf)) or not(0<eaf<1 and 0<gmaf<=.5 and 0<qmaf<=.5):\n        raise ValueError("INVALID_MAF_OR_EAF")\n    if abs(gmaf-min(eaf,1-eaf))>1e-9:\n        raise ValueError("GWAS_EAF_MAF_INCONSISTENT")\n    parts=z["variant_id"].split(":")\n    if len(parts)!=4 or not parts[1].isdigit() or parts[2] not in ("A","C","G","T") or parts[3] not in ("A","C","G","T"):\n        raise ValueError("GRCH37_GWAS_VARIANT_ID_INVALID")\n    return {\n        "is_palindromic":int({ref,alt} in ({"A","T"},{"C","G"})),\n        "is_palindromic_ambig_eaf":int({ref,alt} in ({"A","T"},{"C","G"}) and .4<=eaf<=.6),\n        "is_gwas_gtEx_maf_diff_gt_0_1":int(abs(gmaf-qmaf)>.1),\n        "is_gwas_gtEx_maf_diff_gt_0_2":int(abs(gmaf-qmaf)>.2),\n    }\n\ndef run(index,inputs,out):\n    if out.exists():raise FileExistsError("REFUSE_OLD_OUTPUT")\n    rows=data(index)\n    keys=set()\n    if len(rows)!=646:raise ValueError("EXPECTED_ORIGINAL_646_SOURCE_INDEX")\n    results=[]\n    for rec in rows:\n        locus,dataset,gene=tuple(rec[k] for k in ("locus","dataset_key","gene_base"))\n        key=(locus,dataset,gene)\n        if key in keys:raise ValueError("DUPLICATED_ORIGINAL_COMPOSITE_KEY")\n        keys.add(key)\n        name="__".join(key)+".tsv"\n        if Path(rec["file"]).name!=name:raise ValueError("INDEX_NAME_MISMATCH")\n        p=inputs/name\n        item=dict(locus=locus,dataset_key=dataset,gene_base=gene,\n                  source_filename=name,status="MISSING_INPUT",reason="LOCAL_SNP_TSV_NOT_CACHED",\n                  n_rows=0,n_unique_match_keys=0,n_palindromic=0,\n                  n_palindromic_eaf_in_0_4_to_0_6=0,\n                  n_maf_diff_gt_0_1=0,n_maf_diff_gt_0_2=0,\n                  source_sha256="",\n                  exact_GRCh38_GTeX_ID_tag="NOT_ASSESSABLE",\n                  GWAS_effect_allele_direction="EXTERNAL_ORIGINAL_GWAS_SCHEMA_REQUIRED",\n                  GRCh37_GRCh38_liftover="NOT_INDEPENDENTLY_VERIFIED",\n                  GTEx_ancestry_matched_LD="NOT_PROVIDED")\n        if p.exists():\n            try:\n                snps=data(p)\n                if len(snps)!=int(rec["nsnps"]):\n                    raise ValueError("ROW_COUNT_NOT_INDEX")\n                if any((z["locus"],z["dataset_key"],z["gene_base"])!=key for z in snps):\n                    raise ValueError("INTERNAL_SOURCE_KEY_MISMATCH")\n                if len({z["match_key"] for z in snps})!=len(snps):\n                    raise ValueError("DUPLICATED_GRCH38_SNP_KEY")\n                flags=[parse_record(z) for z in snps]\n                if len(flags)!=len(snps):raise ValueError("INCOMPLETE_SOURCE_AUDIT")\n                item.update(status="PASS_STRUCTURAL_ONLY",reason="",\n                            n_rows=len(snps),n_unique_match_keys=len(snps),\n                            n_palindromic=sum(z["is_palindromic"] for z in flags),\n                            n_palindromic_eaf_in_0_4_to_0_6=sum(z["is_palindromic_ambig_eaf"] for z in flags),\n                            n_maf_diff_gt_0_1=sum(z["is_gwas_gtEx_maf_diff_gt_0_1"] for z in flags),\n                            n_maf_diff_gt_0_2=sum(z["is_gwas_gtEx_maf_diff_gt_0_2"] for z in flags),\n                            source_sha256=filehash(p),\n                            exact_GRCh38_GTeX_ID_tag="EXACT_REF_ALT_GRCH38_REPRODUCED")\n            except Exception as e:\n                item.update(status="FAIL",reason=str(e)[:300])\n        results.append(item)\n    if len(results)!=646:raise ValueError("DENOMINATOR_LOST")\n    counts=dict(collections.Counter(z["status"] for z in results))\n    assessed=[z for z in results if z["status"]=="PASS_STRUCTURAL_ONLY"]\n    overall=dict(\n        schema="IS_PHASE10_11_G1_SOURCE_ALLELE_STRUCTURAL_V1",\n        total_indexed=646,\n        status_counts=counts,\n        total_gene_snp_records=sum(z["n_rows"] for z in assessed),\n        total_palindromic=sum(z["n_palindromic"] for z in assessed),\n        total_palindromic_in_eaf_0_4_to_0_6=sum(z["n_palindromic_eaf_in_0_4_to_0_6"] for z in assessed),\n        total_gwas_gtEx_maf_diff_gt_0_1=sum(z["n_maf_diff_gt_0_1"] for z in assessed),\n        total_gwas_gtEx_maf_diff_gt_0_2=sum(z["n_maf_diff_gt_0_2"] for z in assessed),\n        qtl_GRCh38_variant_internal_structure="PASS" if not counts.get("FAIL") else "FAIL",\n        gwas_effect_allele_direction="UNVERIFIED_NEEDS_ORIGINAL_BBJ_GWAS_EFFECT_ALLELE_CODEBOOK",\n        gwas_GRCh37_to_qtl_GRCh38_liftover="UNVERIFIED_NEEDS_INDEPENDENT_REFERENCE_MAPPING",\n        gtEx_cohort_matched_LD="UNVERIFIED",\n        causal_IS_mechanism="NOT_ESTABLISHED",\n        interpretation="Repeated gene-SNP rows, not unique SNP count or distinct association signals")\n    out.mkdir(parents=True)\n    with (out/"IS_G1_SOURCE_STRUCTURAL_STATUS_646.tsv").open("w",newline="",encoding="utf8") as f:\n        w=csv.DictWriter(f,fieldnames=list(results[0]),delimiter="\\t")\n        w.writeheader();w.writerows(results)\n    (out/"IS_G1_SOURCE_STRUCTURAL_SUMMARY.json").write_text(json.dumps(overall,indent=2,ensure_ascii=False)+"\\n")\n    (out/"IS_G1_METHOD_BOUNDARY.md").write_text(\n        "# IS GWAS/GTEx source G1 structural harmonization QC\\n\\n"\n        f"- Assessed {counts.get(\'PASS_STRUCTURAL_ONLY\',0)} of 646 source assays from local cache; unassessed {counts.get(\'MISSING_INPUT\',0)} are not interpreted as PASS.\\n"\n        f"- Input gene–SNP rows assessed: {overall[\'total_gene_snp_records\']} (SNPs repeated across tissues/genes).\\n"\n        "- GRCh38 match_key equals original GTEx variant encoding and GRCh38 REF/ALT fields; GTEx reported effect/SE/MAF preserved; GWAS MAF is internally consistent with GWAS EAF.\\n"\n        "- Nonzero number of palindromic or ancestry-differing MAF variants is NOT itself evidence of allele flip. Neither ref/alt identity nor GWAS MAF demonstrates original GWAS effect-allele coding.\\n"\n        "- Original BBJ GWAS variant-specific effect allele and independently verified GRCh37->38 liftover remain external missing evidence. Joint GTEx ancestry matched molecular LD also remains unavailable.\\n"\n        "- SOURCE G1 STRUCTURAL PASS does not imply causal gene validity, reciprocal allelic effects, or cohort genotype harmonization.\\n",\n        encoding="utf8")\n    print(json.dumps(overall,ensure_ascii=False))\n    return overall\n\nif __name__=="__main__":\n    p=argparse.ArgumentParser()\n    p.add_argument("--index",type=Path,required=True)\n    p.add_argument("--inputs",type=Path,required=True)\n    p.add_argument("--out",type=Path,required=True)\n    a=p.parse_args()\n    run(a.index,a.inputs,a.out)\n'
G1_SCRIPT=WORK/"audit_is_g1_source_alleles.py"
G1_SCRIPT.write_text(G1_CODE,encoding="utf8")
G1_OUTPUT=OUTPUT/"G1_ORIGINAL_ALLELE_STRUCTURAL_V1"
g1_proc=subprocess.run(["python3",str(G1_SCRIPT),
     "--index",str(INDEX),"--inputs",str(INPUT),"--out",str(G1_OUTPUT)],
     capture_output=True,text=True,check=False)
print("G1_STRUCTURAL_AUDIT_PROCESS_EXIT=",g1_proc.returncode)
print(g1_proc.stdout[-4000:])
if g1_proc.returncode: raise RuntimeError(g1_proc.stderr[-5000:])
G1_SUMMARY=json.loads((G1_OUTPUT/"IS_G1_SOURCE_STRUCTURAL_SUMMARY.json").read_text(encoding="utf8"))
G1_COUNTS=G1_SUMMARY["status_counts"]
G1_STATUS=("G1_STRUCTURAL_646_PASS_EXTERNAL_EFFECT_ALLELE_UNVERIFIED" if G1_COUNTS=={"PASS_STRUCTURAL_ONLY":646} else "G1_STRUCTURAL_INCOMPLETE_OR_FAIL")
print("G1_SOURCE_QC_STATUS=",G1_STATUS)
manifest={
    "schema":"IS_PHASE10_11_646_QTL_N_SCALAR_COLAB_V1",
    "source":"ARCHIVED_FOUR_BBJ_LOCI_GTEX_NOT_EXPANDED_80",
    "original_code_baseline_priors":{"p1":1e-4,"p2":1e-4,"p12":1e-5},
    "R_package_coloc":"5.2.3",
    "sensitivity_scenarios":["ORIGINAL_FIRST_EQ_MAX","MEDIAN_SNP_N","MIN_SNP_N"],
    "n_assumption_status":"THREE_SCALAR_N_STRESS_NOT_SNP_SPECIFIC_MODEL",
    "g1_source_audit_status":G1_STATUS,
    "g1_source_audit_counts":G1_COUNTS,
    "g1_script_sha256":sha(G1_SCRIPT),
    "inputs_available_before_run":n_available,
    "counts":counts,
    "status":"FULL_646_SCALAR_N_STRESS_COMPLETE" if counts["PASS"]==646 else "PARTIAL_OR_FAILED",
    "evidence_boundary":"N_MIN_MEDIAN_FIRST_SCALAR_STRESS_NOT_PER_SNP_N_MODEL_OR_CAUSAL_GENE",
    "source_sha256":{"original_index":sha(INDEX),"original_master":sha(MASTER),
                     "embedded_R_script":sha(script),"output_status":sha(status)},
    "runtime_seconds":round(time.time()-start,2)}
(OUTPUT/"IS_646_N_SCALAR_COLAB_MANIFEST.json").write_text(
  json.dumps(manifest,indent=2,ensure_ascii=False)+"\n",encoding="utf8")
shutil.copyfile(RUNLOG,OUTPUT/"IS_646_N_SCALAR_FULL_RUN_LOG.txt")
print(json.dumps(manifest,indent=2,ensure_ascii=False))
print("IS_PHASE10_11_646_QTL_N_STRESS=RUN_ENDED_AND_STATUS_ACCOUNTED")
